<a href="https://colab.research.google.com/github/anshul202/super_resolution_dark/blob/main/Super_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install unrar

!unrar x "/content/drive/MyDrive/darkSR_dataset.rar" "/content/darkSR_dataset/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Using cached unrar-0.4-py3-none-any.whl.metadata (3.0 kB)
Using cached unrar-0.4-py3-none-any.whl (25 kB)

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/darkSR_dataset.rar

Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0002.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0051.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0058.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0118.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0133.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a0139.npy       0%  OK 
Extracting  /content/darkSR_dataset/Mixed All/TESTING/GT_ISP/a01

In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

class DarkSRDataset(Dataset):
    def __init__(self, root_dir):
        self.gt_dir = os.path.join(root_dir, 'GT_ISP')
        self.lr_isp_dir = os.path.join(root_dir, 'LR_ISP')
        self.lr_raw_dir = os.path.join(root_dir, 'LR_RAW')

        self.file_names = sorted(os.listdir(self.gt_dir))

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        file_name = self.file_names[idx]

        gt_isp = np.load(os.path.join(self.gt_dir, file_name))
        lr_isp = np.load(os.path.join(self.lr_isp_dir, file_name))
        lr_raw = np.load(os.path.join(self.lr_raw_dir, file_name))

        gt_isp = torch.from_numpy(gt_isp).float().permute(2, 0, 1)
        lr_isp = torch.from_numpy(lr_isp).float().permute(2, 0, 1)
        lr_raw = torch.from_numpy(lr_raw).float().permute(2, 0, 1)

        return lr_raw, lr_isp, gt_isp


dataset = DarkSRDataset(root_dir='/content/darkSR_dataset/')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [ ]:
import torch.nn as nn

class HybridTransformerCNN(nn.Module):
    def __init__(self, upscale_factor=2):
        super(HybridTransformerCNN, self).__init__()


        self.cnn_extractor = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1)
        )


        encoder_layer = nn.TransformerEncoderLayer(d_model=64, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.upsampler = nn.Sequential(
            nn.Conv2d(64, 64 * (upscale_factor ** 2), kernel_size=3, padding=1),
            nn.PixelShuffle(upscale_factor),
            nn.Conv2d(64, 3, kernel_size=3, padding=1) # Final output to 3-channel RGB
        )

    def forward(self, lr_raw, lr_isp):
        # Concatenate RAW and ISP inputs along the channel dimension
        x = torch.cat((lr_raw, lr_isp), dim=1)

        # Extract local structural features
        x = self.cnn_extractor(x)

        # Transformers require sequential data, so we flatten spatial dimensions
        # B, C, H, W -> B, H*W, C
        b, c, h, w = x.size()
        x_flat = x.view(b, c, -1).permute(0, 2, 1)

        x_trans = self.transformer(x_flat)

        # Reshape back to 2D image feature map format (B, C, H, W)
        x = x_trans.permute(0, 2, 1).view(b, c, h, w)

        # Upsample to high resolution
        out = self.upsampler(x)
        return out

# Initialize the model and move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridTransformerCNN(upscale_factor=2).to(device)

In [ ]:
import torch.optim as optim

criterion = nn.L1Loss() # L1 Loss (or Charbonnier) is preferred over MSE for images
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Example Training Loop for 1 epoch
model.train()
for batch_idx, (lr_raw, lr_isp, gt_isp) in enumerate(dataloader):
    # Move data to GPU
    lr_raw = lr_raw.to(device)
    lr_isp = lr_isp.to(device)
    gt_isp = gt_isp.to(device)

    # Zero gradients
    optimizer.zero_grad()

    # Forward pass
    output = model(lr_raw, lr_isp)

    # Calculate loss
    loss = criterion(output, gt_isp)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    if batch_idx % 10 == 0:
        print(f"Batch {batch_idx} | Loss: {loss.item():.4f}")